# Hybrid Item-Based CF with Metadata (MovieLens + TMDB + Tags)

이 노트북은 이전 전처리 결과와 새로 제공된 `links.csv`를 이용해서
**평점 + 태그 + 메타데이터(장르/연도/예산/인기도 등)**를 모두 활용하는
하이브리드 Item-Based 협업필터링을 구현한다.

### 사용 데이터
- `processed/ratings_cleaned.csv` : user–movie 평점 + user mean + centered rating
- `processed/movie_tag_corpus.csv` : movieId 기준 태그 문서 (전처리 완료)
- `processed/movies_cleaned.csv` : tmdbId 기준 영화 메타데이터 (전처리 완료)
- `links.csv` : MovieLens `movieId` ↔ TMDB `tmdbId` 매핑
- (선택) `movies_refined.csv` : `movieId` ↔ `title` 매핑 (있으면 결과에 title 붙이기)

### 목표
1. **벡터화**
   - Item–User (평점) → centered rating 행렬
   - Item–Tag       → TF-IDF 벡터
   - Item–Metadata  → 장르(one-hot/multi-hot) + 수치형 feature 정규화
2. **유사도 계산**
   - 평점 기반 cosine + co-rating shrinkage
   - 태그 기반 cosine (TF-IDF)
   - 메타데이터 기반 cosine
   - 세 유사도를 가중 합으로 묶은 hybrid similarity
3. **dot product 기반 예측 (벡터화/병렬화)**
   - `R_centered @ S_hybrid` 형태의 행렬 곱으로 모든 (user, item) 예측
4. **추천 함수 구현**
   - 아직 보지 않은 영화 중 상위 N개 추천
   - 결과를 표 형태로 정돈해서 출력


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

# =====================================================================
# 경로 설정
# =====================================================================
DATA_DIR = Path('.')
PROCESSED_DIR = DATA_DIR / 'processed'

RATINGS_CLEANED_PATH = PROCESSED_DIR / 'ratings_cleaned.csv'
TAG_CORPUS_PATH = PROCESSED_DIR / 'movie_tag_corpus.csv'
MOVIES_CLEANED_PATH = PROCESSED_DIR / 'movies_cleaned.csv'

LINKS_PATH = DATA_DIR / 'links.csv'
MOVIES_REFINED_PATH = DATA_DIR / 'movies_refined.csv'  # 선택: title 매핑용

print('ratings_cleaned:', RATINGS_CLEANED_PATH.resolve())
print('movie_tag_corpus:', TAG_CORPUS_PATH.resolve())
print('movies_cleaned:', MOVIES_CLEANED_PATH.resolve())
print('links:', LINKS_PATH.resolve())
print('movies_refined(optional):', MOVIES_REFINED_PATH.resolve())


ratings_cleaned: C:\Users\gram\Downloads\item_based-recommendation-system\processed\ratings_cleaned.csv
movie_tag_corpus: C:\Users\gram\Downloads\item_based-recommendation-system\processed\movie_tag_corpus.csv
movies_cleaned: C:\Users\gram\Downloads\item_based-recommendation-system\processed\movies_cleaned.csv
links: C:\Users\gram\Downloads\item_based-recommendation-system\links.csv
movies_refined(optional): C:\Users\gram\Downloads\item_based-recommendation-system\movies_refined.csv


## 1. 데이터 로딩 및 ID 정합성 맞추기

1. `ratings_cleaned`에서 **추천의 기준 공간 = movieId**를 잡는다.
2. `movie_tag_corpus`는 이미 movieId 기준이므로 바로 사용 가능.
3. `movies_cleaned`는 tmdbId 기준이라 직접 쓸 수 없고, `links.csv`를 통해 movieId로 매핑해야 한다.


In [ ]:
# =====================================================================
# 1-1. ratings_cleaned 로딩
# =====================================================================
ratings = pd.read_csv(RATINGS_CLEANED_PATH)
print('ratings_cleaned.shape:', ratings.shape)
ratings.head()


ratings_cleaned.shape: (100836, 7)


,userId,movieId,rating,rating_datetime,rating_user_mean,rating_centered,timestamp
0,1,1,4.0,2000-07-30 18:45:03,4.366379,-0.366379,964982703
1,1,3,4.0,2000-07-30 18:20:47,4.366379,-0.366379,964981247
2,1,6,4.0,2000-07-30 18:37:04,4.366379,-0.366379,964982224
3,1,47,5.0,2000-07-30 19:03:35,4.366379,0.633621,964983815
4,1,50,5.0,2000-07-30 18:48:51,4.366379,0.633621,964982931


In [ ]:
# =====================================================================
# 1-2. movie_tag_corpus 로딩
# =====================================================================
movie_tag_corpus = pd.read_csv(TAG_CORPUS_PATH)
print('movie_tag_corpus.shape:', movie_tag_corpus.shape)
movie_tag_corpus.head()


movie_tag_corpus.shape: (1572, 2)


,movieId,tags_joined
0,1,fun pixar
1,2,fantasy game magic board game robin williams
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [ ]:
# =====================================================================
# 1-3. movies_cleaned (tmdbId 기반 메타데이터) 로딩
# =====================================================================
movies_cleaned = pd.read_csv(MOVIES_CLEANED_PATH)
print('movies_cleaned.shape:', movies_cleaned.shape)
movies_cleaned.head()


movies_cleaned.shape: (45435, 14)


,tmdbId,title,original_title,original_language,release_date,release_year,runtime,vote_average,vote_count,popularity,log1p_budget,log1p_revenue,genres_list,main_genre
0,862,Toy Story,Toy Story,en,1995-10-30,1995.0,81.0,7.7,5415.0,21.946943,17.216708,19.738573,"['Animation', 'Comedy', 'Family']",Animation
1,8844,Jumanji,Jumanji,en,1995-12-15,1995.0,104.0,6.9,2413.0,17.015539,17.989898,19.386893,"['Adventure', 'Fantasy', 'Family']",Adventure
2,15602,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,1995.0,101.0,6.5,92.0,11.712900,0.000000,0.000000,"['Romance', 'Comedy']",Romance
3,31357,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,1995.0,127.0,6.1,34.0,3.859495,16.588099,18.215526,"['Comedy', 'Drama', 'Romance']",Comedy
4,11862,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,1995.0,106.0,5.7,173.0,8.387519,0.000000,18.153832,['Comedy'],Comedy


In [ ]:
# =====================================================================
# 1-4. links.csv 로딩 및 tmdbId 정리
# =====================================================================
links = pd.read_csv(LINKS_PATH)
print('links.shape:', links.shape)
links.head()


links.shape: (9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [ ]:
# tmdbId를 숫자로 변환 (일부 NaN/비숫자 값 제거)
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
before = len(links)
links = links.dropna(subset=['tmdbId'])
after = len(links)
links['tmdbId'] = links['tmdbId'].astype(int)

print(f'tmdbId 결측/비숫자 제거: {before - after} rows 제거, 남은 rows = {after}')
links.head()


tmdbId 결측/비숫자 제거: 8 rows 제거, 남은 rows = 9734


,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [ ]:
# =====================================================================
# 1-5. movies_cleaned (tmdbId) ↔ links (movieId, tmdbId) 매핑
# =====================================================================
# 목표: movies_cleaned에 movieId를 붙여서, 메타데이터를 movieId 기준으로 정렬할 수 있도록 만드는 것.
meta_with_movieId = pd.merge(
    movies_cleaned,
    links[['movieId', 'tmdbId']],
    on='tmdbId',
    how='inner'
)

print('meta_with_movieId.shape:', meta_with_movieId.shape)
meta_with_movieId[['movieId', 'title', 'genres_list', 'release_year']].head()


meta_with_movieId.shape: (9545, 15)


,movieId,title,genres_list,release_year
0,1,Toy Story,"['Animation', 'Comedy', 'Family']",1995.0
1,2,Jumanji,"['Adventure', 'Fantasy', 'Family']",1995.0
2,3,Grumpier Old Men,"['Romance', 'Comedy']",1995.0
3,4,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",1995.0
4,5,Father of the Bride Part II,['Comedy'],1995.0


In [ ]:
# =====================================================================
# 1-6. 추천 대상 공간: ratings에 등장하는 movieId만 사용
# =====================================================================
unique_movie_ids = np.sort(ratings['movieId'].unique())
unique_user_ids = np.sort(ratings['userId'].unique())

n_movies = len(unique_movie_ids)
n_users = len(unique_user_ids)

movie_id_to_index = {mid: idx for idx, mid in enumerate(unique_movie_ids)}
user_id_to_index = {uid: idx for idx, uid in enumerate(unique_user_ids)}

print('고유 movie 수:', n_movies)
print('고유 user 수:', n_users)

# 메타데이터도 ratings에 등장하는 movieId만 사용 (그 외는 추천 대상 밖이라고 보고 제외)
meta_movies = meta_with_movieId[meta_with_movieId['movieId'].isin(unique_movie_ids)].copy()
meta_movies['movie_index'] = meta_movies['movieId'].map(movie_id_to_index)
meta_movies = meta_movies.dropna(subset=['movie_index']).sort_values('movie_index')
meta_movies['movie_index'] = meta_movies['movie_index'].astype(int)

print('meta_movies.shape:', meta_movies.shape)
meta_movies[['movieId', 'movie_index', 'title', 'genres_list']].head()


고유 movie 수: 9724
고유 user 수: 610
meta_movies.shape: (9527, 16)


,movieId,movie_index,title,genres_list
0,1,0,Toy Story,"['Animation', 'Comedy', 'Family']"
1,2,1,Jumanji,"['Adventure', 'Fantasy', 'Family']"
2,3,2,Grumpier Old Men,"['Romance', 'Comedy']"
3,4,3,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']"
4,5,4,Father of the Bride Part II,['Comedy']


여기까지로 **모든 데이터의 기준 축을 `movieId`로 통일**했다.
- 평점/태그/메타데이터 모두 같은 item-space(movieId) 안에서 다룰 수 있게 된다.


## 2. 벡터화 (item–user, item–tag, item–metadata)

### 2-1. Item–User: centered rating 행렬
- `ratings_cleaned`의 `rating_centered`를 사용해 (user × item) 희소 행렬을 만든다.
- 관측되지 않은 평점은 0으로 두는데, centered rating에서는 "평가 안 함"이 자연스럽게 0이 되며,
  cosine similarity에서 0인 차원은 유사도에 영향을 주지 않는다.

### 2-2. Item–Tag: TF-IDF
- `movie_tag_corpus`의 `tags_joined`를 TF-IDF로 벡터화.
- item–tag 관계를 단어 빈도 기반으로 벡터화하여, 콘텐츠 유사도를 측정할 수 있게 한다.

### 2-3. Item–Metadata: 장르 one-hot + 수치형 feature 정규화
- `genres_list`는 MultiLabelBinarizer로 multi-hot 인코딩 (장르 one-hot의 확장)
- `release_year`, `runtime`, `vote_average`, `vote_count`, `popularity`, `log1p_budget`, `log1p_revenue` 등을
  StandardScaler로 정규화해 하나의 메타데이터 feature 벡터로 만든다.


In [ ]:
# =====================================================================
# 2-1. centered rating 행렬 R_centered (user x item)
# =====================================================================
row_indices = ratings['userId'].map(user_id_to_index).to_numpy()
col_indices = ratings['movieId'].map(movie_id_to_index).to_numpy()
values = ratings['rating_centered'].to_numpy(dtype=np.float32)

R_centered = sparse.csr_matrix((values, (row_indices, col_indices)), shape=(n_users, n_movies))
print('R_centered shape:', R_centered.shape)
R_centered


R_centered shape: (610, 9724)


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 100836 stored elements and shape (610, 9724)>

In [ ]:
# =====================================================================
# 2-2. 태그 TF-IDF 행렬 (movies_with_tags x tag_dim)
# =====================================================================
tag_movies = movie_tag_corpus[movie_tag_corpus['movieId'].isin(unique_movie_ids)].copy()
tag_movies['movie_index'] = tag_movies['movieId'].map(movie_id_to_index)
tag_movies = tag_movies.dropna(subset=['movie_index']).sort_values('movie_index')
tag_movies['movie_index'] = tag_movies['movie_index'].astype(int)

print('tag_movies.shape:', tag_movies.shape)

tfidf = TfidfVectorizer(
    max_features=5000,   # 벡터 차원 제한
    ngram_range=(1, 2),  # 1-gram + 2-gram
    min_df=2             # 너무 희귀한 태그 제거
)
tag_texts = tag_movies['tags_joined'].fillna('')
X_tag = tfidf.fit_transform(tag_texts)

print('X_tag shape:', X_tag.shape)


tag_movies.shape: (1554, 3)
X_tag shape: (1554, 1109)


In [ ]:
# =====================================================================
# 2-3. 메타데이터 벡터화 (장르 one-hot + 수치형 스케일링)
# =====================================================================
# 1) 장르 multi-hot 인코딩
mlb = MultiLabelBinarizer()
genres_list = meta_movies['genres_list'].apply(lambda x: [] if pd.isna(x) else eval(x) if isinstance(x, str) and x.startswith('[') else x)
# 위 람다는:
# - NaN이면 빈 리스트
# - 문자열이면 eval로 리스트로 변환 (01 전처리에서 이미 리스트로 저장했다면 그대로 사용)

genres_matrix = mlb.fit_transform(genres_list)
print('genres_matrix shape:', genres_matrix.shape)
print('장르 개수:', len(mlb.classes_))

# 2) 수치형 feature 선택 및 결측치 처리
num_cols = ['release_year', 'runtime', 'vote_average', 'vote_count',
            'popularity', 'log1p_budget', 'log1p_revenue']

num_df = meta_movies[num_cols].copy()

# 간단한 전략: 각 feature의 중앙값(median)으로 결측치 채우기
for c in num_cols:
    median_val = num_df[c].median()
    num_df[c] = num_df[c].fillna(median_val)

# 3) StandardScaler로 정규화
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num_df.values)

print('num_scaled shape:', num_scaled.shape)

# 4) 장르 multi-hot + 수치형 scaled를 하나의 메타데이터 feature 벡터로 결합
# (데이터 크기가 크지 않다고 가정하고 dense 사용. 실제 서비스라면 sparse 조합 고려)
X_meta = np.hstack([genres_matrix.astype(np.float32), num_scaled.astype(np.float32)])
print('X_meta shape:', X_meta.shape)


genres_matrix shape: (9527, 20)
장르 개수: 20
num_scaled shape: (9527, 7)
X_meta shape: (9527, 27)


- `X_tag` : 태그 기반 텍스트 feature (item–tag 관계 벡터화)
- `X_meta`: 장르 + 수치형 메타데이터 feature (item–metadata 관계 벡터화)

둘 다 **item-space(movieId)** 기준으로 정렬되는 것이 중요하다.
위 코드에서는 `movie_index`를 통해 ratings의 movieId 순서와 맞춰두었다.


## 3. 아이템–아이템 유사도 계산

### 3-1. 평점 기반 유사도 (cosine + shrinkage)
- centered rating을 이용해 item–user 행렬에서 cosine similarity 계산.
- 공통 평가자 수가 적을수록 유사도가 과대평가되는 문제를 **co-rating count 기반 shrinkage**로 완화.

### 3-2. 태그 기반 유사도
- TF-IDF 벡터 `X_tag`에 cosine similarity 적용.

### 3-3. 메타데이터 기반 유사도
- `X_meta`에 cosine similarity 적용.

### 3-4. Hybrid 결합
- 세 유사도를 `w_rating`, `w_tag`, `w_meta`로 가중합.
- 유사도 행렬에서 대각(자기 자신) 항은 0으로 만들어 이웃 후보에서 제외.


In [ ]:
# =====================================================================
# 3-1. 평점 기반 유사도 (cosine on R_item) + shrinkage
# =====================================================================
R_item = R_centered.T  # shape: (n_movies, n_users)

S_rating = cosine_similarity(R_item)
np.fill_diagonal(S_rating, 0.0)

print('S_rating shape:', S_rating.shape)

# co-rating count 행렬 C 계산
A = R_centered.copy()
A.data = np.ones_like(A.data)
C = (A.T @ A).toarray().astype(np.float32)  # shape: (n_movies, n_movies)

LAMBDA = 20.0  # shrinkage 강도
shrinkage_factor = C / (C + LAMBDA + 1e-8)
S_rating_shrunk = S_rating * shrinkage_factor

print('C[:5,:5]:\n', C[:5, :5])
print('shrinkage_factor[:5,:5]:\n', shrinkage_factor[:5, :5])


S_rating shape: (9724, 9724)
C[:5,:5]:
 [[215.  68.  32.   2.  32.]
 [ 68. 110.  26.   3.  22.]
 [ 32.  26.  52.   1.  19.]
 [  2.   3.   1.   7.   3.]
 [ 32.  22.  19.   3.  49.]]
shrinkage_factor[:5,:5]:
 [[0.9148936  0.77272725 0.61538464 0.09090909 0.61538464]
 [0.77272725 0.84615386 0.5652174  0.13043478 0.52380955]
 [0.61538464 0.5652174  0.7222222  0.04761905 0.4871795 ]
 [0.09090909 0.13043478 0.04761905 0.25925925 0.13043478]
 [0.61538464 0.52380955 0.4871795  0.13043478 0.71014494]]


In [ ]:
# =====================================================================
# 3-2. 태그 기반 유사도 (cosine on X_tag) → 전체 movie space로 확장
# =====================================================================
if X_tag.shape[0] > 0:
    S_tag_sub = cosine_similarity(X_tag)
    np.fill_diagonal(S_tag_sub, 0.0)
    print('S_tag_sub shape:', S_tag_sub.shape)
    
    S_tag = np.zeros((n_movies, n_movies), dtype=np.float32)
    idx_tag = tag_movies['movie_index'].to_numpy()
    for i, gi in enumerate(idx_tag):
        S_tag[np.ix_([gi], idx_tag)] = S_tag_sub[i]
else:
    S_tag = np.zeros((n_movies, n_movies), dtype=np.float32)
    print('태그 기반 유사도를 계산할 데이터가 없어, S_tag는 0 행렬입니다.')


S_tag_sub shape: (1554, 1554)


In [ ]:
# =====================================================================
# 3-3. 메타데이터 기반 유사도 (cosine on X_meta) → 전체 movie space로 확장
# =====================================================================
if X_meta.shape[0] > 0:
    S_meta_sub = cosine_similarity(X_meta)
    np.fill_diagonal(S_meta_sub, 0.0)
    print('S_meta_sub shape:', S_meta_sub.shape)
    
    S_meta = np.zeros((n_movies, n_movies), dtype=np.float32)
    idx_meta = meta_movies['movie_index'].to_numpy()
    for i, gi in enumerate(idx_meta):
        S_meta[np.ix_([gi], idx_meta)] = S_meta_sub[i]
else:
    S_meta = np.zeros((n_movies, n_movies), dtype=np.float32)
    print('메타데이터 기반 유사도를 계산할 데이터가 없어, S_meta는 0 행렬입니다.')


S_meta_sub shape: (9527, 9527)


In [ ]:
# =====================================================================
# 3-4. Hybrid similarity 결합
# =====================================================================
w_rating = 0.6
w_tag = 0.2
w_meta = 0.2

S_hybrid = w_rating * S_rating_shrunk + w_tag * S_tag + w_meta * S_meta

# 자기 자신과의 유사도는 0으로 설정 (이웃 후보에서 제외)
np.fill_diagonal(S_hybrid, 0.0)

print('S_hybrid shape:', S_hybrid.shape)


S_hybrid shape: (9724, 9724)


여기서 null(평가 없음)을 별도 값으로 바꾸지 않고, **0 그대로 두는 이유**는:
- centered rating에서는 "미평가"가 0으로 표현될 때 cosine similarity가 자연스럽게 작동한다.
- 대신, 정보량이 적은 유사도(공통 평가자 수가 적은 아이템 쌍)는
  shrinkage factor `C / (C + λ)`를 곱해 자동으로 줄였다.
이 방식이 단순히 null을 0이나 다른 값으로 치환하는 것보다 더 안정적인 대안이 된다.


## 4. dot product 기반 전체 예측 행렬 계산

Item-Based CF 예측의 기본식:
\[
  \hat{r}_{u,i} = \bar{r}_u + \frac{\sum_j s_{i,j} (r_{u,j} - \bar{r}_u)}{\sum_j |s_{i,j}|}
\]
- `R_centered`에 이미 \(r_{u,j} - \bar{r}_u\)가 들어 있으므로,
  잔차 부분은 단순히 `R_centered @ S_hybrid`로 계산 가능하다.
- 이는 모든 (user, item)에 대한 예측을 **행렬 곱 한 번**으로 끝내는 것이고,
  내부적으로 BLAS 기반 병렬 연산이 이루어져 효율적이다.


In [ ]:
# =====================================================================
# 4-1. 사용자 평균 벡터 준비
# =====================================================================
user_mean_series = ratings.groupby('userId')['rating_user_mean'].mean()
user_mean_vec = user_mean_series.reindex(unique_user_ids).to_numpy()

print('user_mean_vec shape:', user_mean_vec.shape)
print('user_mean_vec[:5]:', user_mean_vec[:5])


user_mean_vec shape: (610,)
user_mean_vec[:5]: [4.36637931 3.94827586 2.43589744 3.55555556 3.63636364]


In [ ]:
# =====================================================================
# 4-2. R_centered @ S_hybrid → 잔차 예측 행렬
# =====================================================================
Rhat_centered = R_centered @ S_hybrid  # shape: (n_users, n_movies)
print('Rhat_centered shape:', Rhat_centered.shape)


Rhat_centered shape: (610, 9724)


In [ ]:
# =====================================================================
# 4-3. 유사도 정규화 (각 아이템의 유사도 절대값 합으로 나누기)
# =====================================================================
sim_norm = np.abs(S_hybrid).sum(axis=0) + 1e-8  # zero division 방지
Rhat_centered_norm = Rhat_centered / sim_norm

print('sim_norm[:5]:', sim_norm[:5])
print('Rhat_centered_norm shape:', Rhat_centered_norm.shape)


sim_norm[:5]: [545.2588  705.83813 721.24786 613.46216 359.63354]
Rhat_centered_norm shape: (610, 9724)


In [ ]:
# =====================================================================
# 4-4. 사용자 평균을 더해 최종 예측 평점 행렬 Rhat 생성
# =====================================================================
Rhat = Rhat_centered_norm + user_mean_vec.reshape(-1, 1)

print('Rhat shape:', Rhat.shape)
print(pd.Series(Rhat.flatten()).describe())


Rhat shape: (610, 9724)
count    5.931640e+06
mean     3.656896e+00
std      4.804844e-01
min      9.417138e-01
25%      3.359626e+00
50%      3.695780e+00
75%      3.998647e+00
max      5.000000e+00
dtype: float64


이 과정 전체는 파이썬 for-loop 없이 **희소/밀집 행렬 연산**으로 구성되어,
실질적으로는 내부 C/BLAS 레벨에서 병렬 처리된다.


## 5. 추천 함수 구현

특정 사용자 `user_id`에 대해:
1. `Rhat`에서 해당 사용자의 예측 벡터를 가져온다.
2. 이미 평점을 남긴 영화는 제외.
3. 예측 평점이 일정 threshold 이상인 영화 중 상위 N개를 추천.
4. `movies_refined`가 있으면 title을 붙여 결과를 읽기 좋게 만든다.


In [ ]:
# =====================================================================
# 5-1. (선택) title 매핑 정보 로딩
# =====================================================================
if MOVIES_REFINED_PATH.exists():
    movies_refined = pd.read_csv(MOVIES_REFINED_PATH)
    print('movies_refined.shape:', movies_refined.shape)
    print('movies_refined.columns:', movies_refined.columns.tolist())
else:
    movies_refined = None
    print('movies_refined.csv가 없어 movieId만 출력됩니다.')


movies_refined.csv가 없어 movieId만 출력됩니다.


In [ ]:
# =====================================================================
# 5-2. 보조: 인덱스 매핑 함수
# =====================================================================
def get_user_index(user_id: int) -> int:
    if user_id not in user_id_to_index:
        raise ValueError(f'user_id {user_id}는 ratings에 존재하지 않습니다.')
    return user_id_to_index[user_id]

def get_movie_index(movie_id: int) -> int:
    if movie_id not in movie_id_to_index:
        raise ValueError(f'movie_id {movie_id}는 ratings에 존재하지 않습니다.')
    return movie_id_to_index[movie_id]


In [ ]:
# =====================================================================
# 5-3. 추천 함수
# =====================================================================
def recommend_items_for_user(
    user_id: int,
    top_n: int = 10,
    min_pred_rating: float = 3.5
):
    """
    user_id에 대해, 아직 보지 않은 영화 중 예측 평점이 높은 상위 N개를 추천한다.
    
    Parameters
    ----------
    user_id : int
        추천 대상 사용자 ID (ratings.csv 기준)
    top_n : int
        반환할 추천 영화 수
    min_pred_rating : float
        이 값보다 예측 평점이 낮으면 추천 후보에서 제외
    """
    u_idx = get_user_index(user_id)
    user_pred = Rhat[u_idx]  # shape: (n_movies,)
    
    # 이미 본 영화 제외
    watched_movie_ids = ratings.loc[ratings['userId'] == user_id, 'movieId'].unique()
    watched_indices = [movie_id_to_index[mid] for mid in watched_movie_ids if mid in movie_id_to_index]
    
    mask = np.ones(n_movies, dtype=bool)
    mask[watched_indices] = False
    
    # 예측 평점 threshold 적용
    candidate_indices = np.where(mask & (user_pred >= min_pred_rating))[0]
    if len(candidate_indices) == 0:
        print(f"user_id={user_id} 에 추천 후보가 없습니다.")
        return pd.DataFrame()
    
    sorted_candidate_indices = candidate_indices[np.argsort(user_pred[candidate_indices])[::-1]]
    top_indices = sorted_candidate_indices[:top_n]
    
    rec_movie_ids = [unique_movie_ids[i] for i in top_indices]
    rec_pred_ratings = user_pred[top_indices]
    
    result = pd.DataFrame({
        'movieId': rec_movie_ids,
        'pred_rating': rec_pred_ratings
    })
    
    # title 붙이기 (가능한 경우)
    if movies_refined is not None:
        # 파일 구조에 따라 컬럼명 맞추기
        if 'movie_id' in movies_refined.columns:
            tmp_movies = movies_refined[['movie_id', 'title']].rename(columns={'movie_id': 'movieId'})
        elif 'movieId' in movies_refined.columns:
            tmp_movies = movies_refined[['movieId', 'title']]
        else:
            tmp_movies = None
        
        if tmp_movies is not None:
            result = result.merge(tmp_movies, on='movieId', how='left')
    
    cols = ['movieId', 'title', 'pred_rating'] if 'title' in result.columns else ['movieId', 'pred_rating']
    result = result[cols]
    
    return result


In [ ]:
# =====================================================================
# 5-4. 예시 실행
# =====================================================================
example_user_id = int(unique_user_ids[0])
print('예시 user_id:', example_user_id)

rec_df = recommend_items_for_user(example_user_id, top_n=10, min_pred_rating=3.5)
rec_df


예시 user_id: 1


,movieId,pred_rating
0,176579,4.398156
1,176389,4.398156
2,720,4.391896
3,179401,4.388131
4,193565,4.385882
5,1343,4.385786
6,915,4.385567
7,176413,4.385105
8,180231,4.384075
9,177593,4.382273


### 추천 로직이 "풍부한 추천"이 되는 이유

1. **평점 기반 (협업 필터링)**: 비슷한 평점을 준 아이템들의 패턴을 사용.
2. **태그 기반 (콘텐츠)**: 유저가 좋아한 영화와 태그적으로 비슷한 영화가 상위에 올라옴.
3. **메타데이터 기반 (장르/연도/인기도 등)**: 장르/시대/규모가 비슷한 영화가 보완적으로 섞임.

세 유사도를 함께 쓰는 hybrid 구조이기 때문에, 단순히 "평점만 비슷한 아이템"이 아니라
콘텐츠적으로도 유사한 영화들이 추천 리스트에 포함된다.

추가로 실험해볼 수 있는 것들:
- `w_rating`, `w_tag`, `w_meta` 가중치 조절
- `LAMBDA`(shrinkage 강도) 변경
- `min_pred_rating`를 높여 더 강한 필터링
- 결과 리스트에서 장르 다양성 확보를 위한 re-ranking (예: 너무 같은 장르만 몰리는 것 방지)
